In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

load_dotenv(override=True)  # 覆盖环境变量

model = ChatOpenAI(
    model="MiniMax-M3",
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
)

@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str) -> str:
    """
    获取指定公司的股票价格信息

    Args:
        company (str): 公司名称
        timeframe (str): 时间范围(today-今天，weeek-本周，month-本月)
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.00, "weeek": 25, "month": 30},
        "微软公司": {"today": 300.00, "weeek": 28, "month": 35},
        "谷歌公司": {"today": 2800.00, "weeek": 22, "month": 25},
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe, "无数据")
        return f"{company}的股票价格在{timeframe}是{price}美元。"
    else:
        return f"未找到{company}的股票价格信息。"

@tool()
def search_news(company: str) -> str:
    """
    获取指定公司的新闻信息

    Args:
        company (str): 公司名称
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": "苹果公司发布了新款iPhone，市场反应积极。",
        "微软公司": "微软公司宣布收购一家人工智能初创公司。",
        "谷歌公司": "谷歌公司推出了新的搜索算法，提升了搜索体验。",
    }

    return mock_news.get(company, f"未找到{company}的新闻信息。")

model_with_tools = model.bind_tools([search_news, get_stock_price])
message_list = [HumanMessage(content="请告诉我苹果公司今天的股票价格是多少？还有最近有什么新闻")]

while True:
    response = model_with_tools.invoke(message_list)

    message_list.append(response)
    if not response.tool_calls:
        rprint("没有工具调用，结束循环")
        break
    
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            message_list.append(news_result)

for msg in message_list:
    msg.pretty_print()


rprint(message_list)

没有工具调用，结束循环

================================ Human Message =================================

请告诉我苹果公司今天的股票价格是多少？还有最近有什么新闻
================================== Ai Message ==================================

<think>用户想要了解苹果公司今天的股票价格和最近的新闻。我需要调用两个工具来获取这些信息。这些调用是独立的，可以并行执行。</think>

我来帮您查询苹果公司的股票价格和最新新闻。
Tool Calls:
  get_stock_price (call_019feb1a22617502bdaa0f06)
 Call ID: call_019feb1a22617502bdaa0f06
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_019feb1a22617502bdaa0f1f)
 Call ID: call_019feb1a22617502bdaa0f1f
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司的股票价格在today是185.0美元。
================================= Tool Message =================================
Name: search_news

苹果公司发布了新款iPhone，市场反应积极。
================================== Ai Message ==================================

<think>工具调用都成功了，我得到了苹果公司的股票价格和最新新闻。现在我可以整理这些信息回复给用户。</think>

以下是为您查询到的苹果公司相关信息：

## 📈 今日股票价格
**苹果公司（AAPL）今日收盘价：185

[
    HumanMessage(
        content='请告诉我苹果公司今天的股票价格是多少？还有最近有什么新闻',
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='<think>用户想要了解苹果公司今天的股票价格和最近的新闻。我需要调用两个工具来获取这些信息。这些调用
是独立的，可以并行执行。</think>\n\n我来帮您查询苹果公司的股票价格和最新新闻。',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 104,
                'prompt_tokens': 527,
                'total_tokens': 631,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 28,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 526},
                'total_characters': 0
            },
            'model_provider': 'openai',
            'model_name': 'MiniMax-M3',
            'system_fingerprint': None,
            'id': '06c8d349e24dac407fd2d32797e37005',
            'service_tier': 'standard',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019feb1a-17ad-7de3-8493-f8dfce307ce9-0',
        tool_calls=[
            {
                'name': 'get_stock_price',
                'args': {'company': '苹果公司', 'timeframe': 'today'},
                'id': 'call_019feb1a22617502bdaa0f06',
                'type': 'tool_call'
            },
            {
                'name': 'search_news',
                'args': {'company': '苹果公司'},
                'id': 'call_019feb1a22617502bdaa0f1f',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 527,
            'output_tokens': 104,
            'total_tokens': 631,
            'input_token_details': {'cache_read': 526},
            'output_token_details': {'reasoning': 28}
        }
    ),
    ToolMessage(
        content='苹果公司的股票价格在today是185.0美元。',
        name='get_stock_price',
        tool_call_id='call_019feb1a22617502bdaa0f06'
    ),
    ToolMessage(
        content='苹果公司发布了新款iPhone，市场反应积极。',
        name='search_news',
        tool_call_id='call_019feb1a22617502bdaa0f1f'
    ),
    AIMessage(
        content='<think>工具调用都成功了，我得到了苹果公司的股票价格和最新新闻。现在我可以整理这些信息回复给用户。<
/think>\n\n以下是为您查询到的苹果公司相关信息：\n\n## 📈 今日股票价格\n**苹果公司（AAPL）今日收盘价：185.0 
美元**\n\n## 📰 
最新新闻\n**苹果公司发布了新款iPhone，市场反应积极。**\n\n---\n\n如果您需要了解更多详细信息，例如本周或本月的股价走
势，以及更深入的新闻内容分析，请告诉我，我可以进一步为您查询。',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 100,
                'prompt_tokens': 671,
                'total_tokens': 771,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 22,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 536},
                'total_characters': 0
            },
            'model_provider': 'openai',
            'model_name': 'MiniMax-M3',
            'system_fingerprint': None,
            'id': '06c8d34b1b8ed10b2ee9677068a1d183',
            'service_tier': 'standard',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019feb1a-2437-7461-9fda-2cc543359dab-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 671,
            'output_tokens': 100,
            'total_tokens': 771,
            'input_token_details': {'cache_read': 536},
            'output_token_details': {'reasoning': 22}
        }
    )
]